In [130]:
class SyllableAnalyze:
    # 定義合法的音節組件
    INITIALS_TL = {'p', 'ph', 'm', 'b', 't', 'th', 'n', 'l', 'k', 'kh', 'ng', 'g', 'ts', 'tsh', 's', 'j', 'h'}
    INITIALS_POJ = {'p', 'ph', 'm', 'b', 't', 'th', 'n', 'l', 'k', 'kh', 'ng', 'g', 'ch', 'chh', 's', 'j', 'h'}
    VOWELS_TL = {'a', 'i', 'u', 'e', 'oo', 'o'}
    VOWELS_POJ = {'a', 'i', 'u', 'e', 'o͘', 'o'}
    CODAS = {'n','ng', 'm'}
    NASAL_MARKS = {'nn','ⁿ'}
    VOWELIZED_CONSONANTS = {'ng','m'}
    CHECKED_TONES = ('p','t','k','h')
    
    # 母音優先順序
    VOWEL_PRIORITY_TL = {'a':3,
                         'oo':1,
                         'e':1, 
                         'o':1, 
                         'i':0, 
                         'u':0}
    VOWEL_PRIORITY_POJ = {'o':4,
                          'o͘':4,
                          'e':3, 
                          'a':2, 
                          'u':1, 
                          'i':0}
    TONE_MARKS = {
        'a': {'2': 'á', '3': 'à', '5': 'â', '6': 'ǎ', '7': 'ā', '8': 'a̍'},
        'i': {'2': 'í', '3': 'ì', '5': 'î', '6': 'ǐ', '7': 'ī', '8': 'i̍'},
        'u': {'2': 'ú', '3': 'ù', '5': 'û', '6': 'ǔ', '7': 'ū', '8': 'u̍'},
        'e': {'2': 'é', '3': 'è', '5': 'ê', '6': 'ě', '7': 'ē', '8': 'e̍'},
        'oo': {'2': 'óo', '3': 'òo', '5': 'ôo', '6': 'ǒo', '7': 'ōo', '8': 'o̍o'},
        'o': {'2': 'ó', '3': 'ò', '5': 'ô', '6': 'ǒ', '7': 'ō', '8': 'o̍'},
        'ng': {'2': 'ńg', '3': 'ǹg', '5': 'n̂g', '6': 'ňg', '7': 'n̄g', '8': 'n̍g'},
        'm': {'2': 'ḿ', '3': 'm̀', '5': 'm̂', '6': 'm̌', '7': 'm̄', '8': 'm̍'},
        'o͘': {'2': 'ó͘', '3': 'ò͘', '5': 'ô͘', '6': 'ǒ͘', '7': 'ō͘', '8': 'o̍͘'}
    }
    

    def __init__(self, syllable: str, tone: str, is_poj=False):
        """
        初始化拼音音節處理器
        :param syllable: 拼音音節（不含聲調）
        :param tone: 聲調（1-4）
        """
        self.original :str = syllable
        self.tone :str = tone
        self.initial :str = '' #聲母
        self.final :str = '' #整個韻母
        self.vowels :str = '' #元音成分
        self.coda :str = '' #韻尾
        self.nasal :str = '' #鼻化成分
        self.main_vowel :str = ''
        self.checked :bool = False
        self.is_poj :bool = is_poj
        self.is_title :bool = False

    def parse(self):
        """解析音節結構"""
        # 檢查是否為空
        if not self.original:
            raise ValueError("音節不能為空")
        
        if self.original.istitle():
            self.is_title = True
            
        # 找出聲母
        current = self.original.lower()
        if self.is_poj:
            initial_list = self.INITIALS_POJ
        else:
            initial_list = self.INITIALS_TL

        if len(current) > 1: #音節長度 > 1 才有可能是聲母韻母的組合，否則聲母為零聲母
            for initial in sorted(initial_list, key=len, reverse=True):
                if current.startswith(initial):
                    self.initial = initial
                    current = current[len(initial):]
                    break
        if current == '':
            current = self.initial
            self.initial = ''

        self.final = current

        #確認入聲韻尾
        for tone in self.CHECKED_TONES:
            if current.endswith(tone): 
                self.checked = True
                self.coda = tone
                current = current[:-len(tone)]
                break
       
        for mark in self.NASAL_MARKS:
            if current.endswith(mark):
                self.nasal = mark
                current = current[:-len(mark)]
                break

        #確認陽聲韻
        for coda in sorted(self.CODAS,key=len,reverse=True):
            if current.endswith(coda) and len(self.coda) == 0:
                self.coda = coda
                current = current[:-len(coda)]
                break
            
        # #辨認鼻化標記是否被抓到韻尾，調整正確
        # for nasal_mark in self.NASAL_MARKS:
        #     if self.coda == nasal_mark:
        #         self.nasal = nasal_mark
        #         self.coda = ''
        #         break 
            
        self.vowels = current #此時元音部分應已確認
           
        # 檢查元音數量並處理元音化輔音的可能性
        if not 1 <= len(self.vowels) <= 3:
            for consonant in self.VOWELIZED_CONSONANTS:
                if self.coda == consonant:
                    self.vowels = consonant
                    self.coda = ''
                    break

        if len(self.vowels) ==0:
            if self.initial in self.VOWELIZED_CONSONANTS:
                self.vowels = self.initial
                self.initial = ''
                self.final = self.vowels+self.coda
            else:
                raise ValueError(f"母音數量必須在1-3個之間，目前有{len(self.vowels)}個")
            
        # 尋找主要母音
        self.main_vowel = self._find_main_vowel()
        
        return self
        
    def _find_main_vowel(self) -> str:
        """根據優先順序找出主要母音"""
        if not self.vowels:
            raise ValueError("沒有找到母音")
            
        # 根據優先順序表找出優先級最高的母音
        highest_priority = -1
        main_vowel = ''
        
        if self.vowels == 'oo':
            main_vowel = 'oo'
            return main_vowel
        elif self.vowels == 'o͘':
            main_vowel = 'o͘'
            return main_vowel
        
        if self.vowels in self.VOWELIZED_CONSONANTS:
            main_vowel = self.vowels
            return main_vowel

        vowels = list(self.vowels)

        if self.is_poj:
            priority_list = self.VOWEL_PRIORITY_POJ
        else:
            priority_list = self.VOWEL_PRIORITY_TL

        for vowel in vowels:
            priority = priority_list[vowel]
            if priority >= highest_priority: #後來者會替換前者
                highest_priority = priority
                main_vowel = vowel
                               
        return main_vowel
        
    def add_tone_mark(self) -> str:

        if not self.main_vowel or not self.tone:
            raise ValueError("缺少主要母音或聲調信息")
            
        if self.tone not in {'1', '2', '3','4', '5', '6', '7', '8'}:
            raise ValueError("聲調必須是1-8之間的數字")
            
        # 獲取帶聲調的母音
        if self.tone in {'1','4'}:
            toned_vowel = self.main_vowel # 1 4調不用調符
        else:
            toned_vowel = self.TONE_MARKS[self.main_vowel][self.tone]
        
        # 替換原始字符串中的主要母音
        result = self.original.replace(self.main_vowel, toned_vowel)

        return result
    
    def convert(self):
        if not self.is_poj:
            self.initial = self.initial.replace('ts','ch')
            self.final = self.final.replace('nn','ⁿ')
            self.nasal = self.nasal.replace('nn','ⁿ')
            
            if self.final in {'ing','ik'}:
                self.final = self.final.replace('i','e')
                self.vowels = 'e'
                self.main_vowel = 'e'
            
            if self.vowels == 'oo':
                self.final = self.final.replace('oo','o͘')
                self.vowels = 'o͘'
                self.main_vowel = 'o͘'
            if self.vowels in {'ua','ue'}:
                self.final = self.final.replace('u','o')
                self.vowels = self.vowels.replace('u','o')
                self.main_vowel = 'o'
            if self.vowels in {'iu','ui'}:
                self.main_vowel = 'u'
            
            self.original = self.initial+self.final

            if self.is_title:
                self.original = self.original.title()

            self.is_poj = True
        else:
            self.initial = self.initial.replace('ch','ts')
            self.final = self.final.replace('ⁿ','nn')
            self.nasal = self.nasal.replace('ⁿ','nn')
            
            if self.final in {'eng','ek'}:
                self.final = self.final.replace('e','i')
                self.vowels = 'i'
                self.main_vowel = 'i'
            
            if self.vowels == 'o͘':
                self.final = self.final.replace('o͘','oo')
                self.vowels = 'oo'
                self.main_vowel = 'oo'
            if self.vowels in {'oa','oe'}:
                self.final = self.final.replace('o','u')
                self.vowels = self.vowels.replace('o','u')
                if self.vowels == 'ua':
                    self.main_vowel = 'a'
                else:
                    self.main_vowel = 'e'
            if self.vowels == 'ui':
                self.main_vowel = 'i'
            
            self.original = self.initial+self.final

            if self.is_title:
                self.original = self.original.title()
            
            self.is_poj = False
            
        return self

            

In [107]:
import re
import unicodedata

class TaigiModes:
    # 台羅 -> 白話
    MODE_TL2POJ = 0
    # 白話字 -> 台羅
    MODE_POJ2TL = 1
    # 數字標 -> 符號標
    MODE_NO2DIAC = 2
    # 符號標 -> 數字標
    MODE_DIAC2NO = 3

tone_marks:dict = {'0301':'2','0300':'3','0302':'5','030c':'6','0304':'7','030d':'8','030b':'9',
                   '2':'0301','3':'0300','5':'0302','6':'030c','7':'0304','8':'030d','9':'030b'}

def process_im_tsat(syllable: str, tone: str) -> str:
    """
    處理單個拼音音節
    :param syllable: 拼音音節（不含聲調）
    :param tone: 聲調(1-8)
    :return: 處理後的拼音（含變音符號）
    """
    processor = SyllableAnalyze(syllable, tone)
    processor.parse()
    return processor

def diacritic_removal(syllable:str) -> str:
    normalized = unicodedata.normalize('NFD',syllable)
    tone=''
    for idx, letter in enumerate(normalized):
        if unicodedata.category(letter) == 'Mn' and letter != '\u0358' :
            tone = tone_marks[format(ord(letter.lower()),'04x')]
            main_vowel = normalized[idx-1]
            main_vowel_idx = idx-1
    if tone:
        syllable = normalized[:main_vowel_idx]+main_vowel+normalized[main_vowel_idx+2:]
    else:
        if syllable.endswith(SyllableAnalyze.CHECKED_TONES):
            tone = '4'
        else:
            tone = '1'

    result = syllable+tone
    result = unicodedata.normalize("NFC",result)

    return result,syllable,tone

class TaigiConverter:
    def __init__(self) -> None:
        self.mode: int = None
        self.text: str = None
    def update(self, text: str, mode: int) -> "TaigiConverter":
        self.text: str = text
        self.mode: int = mode
        return self
    

    @staticmethod
    def _convert_between_tl_and_poj(text: str) -> str:
        # 請實作
        return text
    
    @staticmethod
    def _convert_between_no_and_diacritics(text: str, mode: int) -> str:
        # 請實作
        pattern = re.compile(r'[a-z0-9A-Z\u0301\u0300\u0302\u030c\u0304\u030d\u030b\-]+')
        target = unicodedata.normalize("NFD",text)
        matches = pattern.findall(target)
        matches = [item for sublist in matches for item in sublist.split('-')]
        for match in sorted(matches, key=len,reverse=True):
            if mode == 2 :
                added = process_im_tsat(match[:-1],match[-1:])
                target = target.replace(match,added)
            else:
                removal = diacritic_removal(match)
                target = target.replace(match, removal)
        text = target
            
        return text

    def convert(self) -> str:
        if 0 <= self.mode < 2:
            return self._convert_between_tl_and_poj(self.text,self.mode)
        if 2 <= self.mode < 4:
            return self._convert_between_no_and_diacritics(self.text, self.mode)
        raise ValueError("Unsupported Mode!")

In [108]:
TESTSTR_TL2POJ = ''''''
    # 白話字 -> 台羅
TESTSTR_POJ2TL = ''''''
    # 數字標 -> 符號標
TESTSTR_NO2DIAC = '''Jip8-tang1 i2-au7, m7-na7 thinn1-khi3 tsuan2-ling2, liu5-hing5-sing3 kam2-moo7 e5 penn7-tok8 ma7 tue3-leh4 khai1-si2 uah8-thiau3, tso7-sing5 put4-tsi2-a2 tse7 hak8-sing1 gin2-a2 kap4 tua7-lang5 tioh8-penn7.'''
    # 符號標 -> 數字標
TESTSTR_DIAC2NO = '''Ji̍p-tang í-āu, m̄-nā thinn-khì tsuán-líng, liû-hîng-sìng kám-mōo ê pēnn-to̍k mā tuè-leh khai-sí ua̍h-thiàu, tsō-sîng put-tsí-á tsē ha̍k-sing gín-á kap tuā-lâng tio̍h-pēnn.'''

In [16]:
# input_text: str = "guá sī ông-iok-tik"
taigi_converter = TaigiConverter()
# taigi_converter.update(TESTSTR_DIAC2NO, TaigiModes.MODE_DIAC2NO)
taigi_converter.update(TEST
                       STR_NO2DIAC, TaigiModes.MODE_NO2DIAC)
output_text: str = taigi_converter.convert()
print(output_text)
# assert(output_text == "gua2 si7 ong5-iok4-tik4")

SyntaxError: invalid syntax. Perhaps you forgot a comma? (3496199703.py, line 4)

##TL POJ轉換試做

In [109]:
syls = ['ngiauh','ńg','n̍gh','mn̂g','tsi̍k','uánn','ue̍h','tsuí','siūnn','íng','tn̂g']

print(f"{'Im-tsat':>10} {'Initial':<7} {"Final":<5} {'Vowels':>6} {'Nasal':<5} {'Coda':<4} {"Tone"}")
for syl in syls:
    imtsat = diacritic_removal(syl)

    test = SyllableAnalyze(imtsat[1],imtsat[2])
    test.parse()

    print(f"{test.add_tone_mark():>10} {test.initial:<7} {test.final:<5} {test.vowels:>6} {test.nasal:<5} {test.coda:<4} {test.tone}")

    test.convert()
    print(f"{test.add_tone_mark():>10} {test.initial:<7} {test.final:<5} {test.vowels:>6} {test.nasal:<5} {test.coda:<4} {test.tone}")
    test.convert()
    print(f"{test.add_tone_mark():>10} {test.initial:<7} {test.final:<5} {test.vowels:>6} {test.nasal:<5} {test.coda:<4} {test.tone}\n")

        


   Im-tsat Initial Final Vowels Nasal Coda Tone
    ngiauh ng      iauh     iau       h    4
    ngiauh ng      iauh     iau       h    4
    ngiauh ng      iauh     iau       h    4

        ńg         ng        ng            2
        ńg         ng        ng            2
        ńg         ng        ng            2

      n̍gh         ngh       ng       h    8
      n̍gh         ngh       ng       h    8
      n̍gh         ngh       ng       h    8

      mn̂g m       ng        ng            5
      mn̂g m       ng        ng            5
      mn̂g m       ng        ng            5

     tsi̍k ts      ik         i       k    8
     che̍k ch      ek         e       k    8
     tsi̍k ts      ik         i       k    8

      uánn         uann      ua nn         2
       óaⁿ         oaⁿ       oa ⁿ          2
      uánn         uann      ua nn         2

      ue̍h         ueh       ue       h    8
      o̍eh         oeh       oe       h    8
      ue̍h         ueh       ue       h    8



##新的轉長字串方法，尚未加入Class

In [135]:
#測試可行的新轉換parsing

import re

def sentence_process(sentence:str,mode:int):


    pattern = re.compile(r'[a-z0-9A-Zⁿo͘\u0301\u0300\u0302\u030c\u0304\u030d\u030b\-]+')
    target = unicodedata.normalize("NFD",sentence)

    words = pattern.findall(target)
    words = [item for sublist in words for item in sublist.split('-')]

    start=0
    end=0

    new = ''

    for word in words:
        end = target.find(word,start)
        new += target[start:end]

        match mode:
            case TaigiModes.MODE_TL2POJ:
                word_num = diacritic_removal(word)
                processed = SyllableAnalyze(word_num[1],word_num[2])
                # print(f"{processed.original}  {processed.is_poj} {processed.original.istitle()}")
                processed.parse()
                print(processed.original)
                processed.convert()
                processed = processed.add_tone_mark()
            case TaigiModes.MODE_POJ2TL:
                word_num = diacritic_removal(word)
                processed = SyllableAnalyze(word_num[1],word_num[2],True)
                processed.parse()
                processed.convert()
                processed = processed.add_tone_mark()
            case TaigiModes.MODE_NO2DIAC:
                processed = SyllableAnalyze(word[:-1],word[-1:])
                processed.parse()
                processed = processed.add_tone_mark()
            case TaigiModes.MODE_DIAC2NO:
                processed = diacritic_removal(word)[0]
        new += processed
        start = end+len(word)

    new+=target[start:]
    return new

v1 = sentence_process(TESTSTR_NO2DIAC,TaigiModes.MODE_NO2DIAC)
print(v1)
v2 = sentence_process(v1,TaigiModes.MODE_DIAC2NO)
print(v2)
v3 = sentence_process(v2,TaigiModes.MODE_NO2DIAC)
print(v3)
v4 = sentence_process(v3,TaigiModes.MODE_TL2POJ)
print(v4)
v5 = sentence_process(v4,TaigiModes.MODE_POJ2TL)
print(v5)



Ji̍p-tang í-āu, m̄-nā thinn-khì tsuán-líng, liû-hîng-sìng kám-mōo ê pēnn-to̍k mā tuè-leh khai-sí ua̍h-thiàu, tsō-sîng put-tsí-á tsē ha̍k-sing gín-á kap tuā-lâng tio̍h-pēnn.
Jip8-tang1 i2-au7, m7-na7 thinn1-khi3 tsuan2-ling2, liu5-hing5-sing3 kam2-moo7 e5 penn7-tok8 ma7 tue3-leh4 khai1-si2 uah8-thiau3, tso7-sing5 put4-tsi2-a2 tse7 hak8-sing1 gin2-a2 kap4 tua7-lang5 tioh8-penn7.
Ji̍p-tang í-āu, m̄-nā thinn-khì tsuán-líng, liû-hîng-sìng kám-mōo ê pēnn-to̍k mā tuè-leh khai-sí ua̍h-thiàu, tsō-sîng put-tsí-á tsē ha̍k-sing gín-á kap tuā-lâng tio̍h-pēnn.
Jip
tang
i
au
m
na
thinn
khi
tsuan
ling
liu
hing
sing
kam
moo
e
penn
tok
ma
tue
leh
khai
si
uah
thiau
tso
sing
put
tsi
a
tse
hak
sing
gin
a
kap
tua
lang
tioh
penn
ji̍p-tang í-āu, m̄-nā thiⁿ-khì chóan-léng, liû-hêng-sèng kám-mō͘ ê pēⁿ-to̍k mā tòe-leh khai-sí o̍ah-thiàu, chō-sêng put-chí-á chē ha̍k-seng gín-á kap tōa-lâng tio̍h-pēⁿ.
ji̍p-tang í-āu, m̄-nā thinn-khì tsuán-líng, liû-hîng-sìng kám-mōo ê pēnn-to̍k mā tuè-leh khai-sí ua̍h-thiàu, tsō-s

In [70]:
a= process_im_tsat("ngiauh",'7')
a

'ngiāuh'